In [254]:
import numpy as np 
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.ticker as ticker
import pickle
import math

from open_iris_client import EyeData, EyesData, ExtraData, Point

root = Path('/home/dan/work/CNS-OpenIrisDAC')


In [ ]:
def edgenerator(filename):
    with open(filename,'rb') as f:
        while True:
            try:
                ed = pickle.load(f)
                yield ed
            except EOFError:
                break       


In [ ]:
def load_cal_data(filename):
    n = 0
    for ed in edgenerator(filename):
        n += 1
    print(f"Found {n} records in {filename}")

    # now allocate arrays and fill them
    datashape = (n,2)
    frame_number_array = np.zeros(n)
    pupil_array = np.zeros(datashape)
    cr_array = np.zeros(datashape)
    p4_array = np.zeros(datashape)
    pupil_scatter_array = np.zeros(datashape)
    cr_scatter_array = np.zeros(datashape)
    p4_scatter_array = np.zeros(datashape)
    scatter_count = 0
    counter = 0
    button_up = True
    button_list=[]  # (index,x,y)
    button_is_pressed = False
    framesig_present = False
    framesig_on = False
    framesig_on_at = 0

    for ed in edgenerator(filename):
        data_ok = True
        if ed.left.pupil.x == 0 or ed.left.pupil.y == 0:
            pupil_array[counter,:] = np.nan
            data_ok = False
        else:
            pupil_array[counter,0] = ed.left.pupil.x
            pupil_array[counter,1] = ed.left.pupil.y
        frame_number_array[counter] = ed.left.frame_number
        if ed.left.cr.x == -100 or ed.left.cr.y == -100:
            cr_array[counter,:] = np.nan
            data_ok = False
        else:
            cr_array[counter,0] = ed.left.cr.x
            cr_array[counter,1] = ed.left.cr.y
        if ed.left.p4.x == 820 or ed.left.p4.y == -100:
            p4_array[counter,:] = np.nan
            data_ok = False
        else:
            p4_array[counter,0] = ed.left.p4.x
            p4_array[counter,1] = ed.left.p4.y

        # if ed.extra.ints[0] != 12:
        #     print(f"frame at {counter}: {ed.extra.ints[0]}")
            
        button_is_pressed = ed.extra.ints[8] & 0x1
        framesig_present = ed.extra.ints[0] & 0x1

        if not framesig_on:
            if framesig_present:
                framesig_on_at = counter
                framesig_on = True
        else:
            # framesig_on is True, meaning the last time through the FRAME signal was present.
            # We only need to check if framesig is not present this time.
            if not framesig_present:
                framesig_on = False

        if button_up:
            if button_is_pressed:
                button_up = False
                if data_ok:
                    button_list.append((counter, ed.extra.doubles[7], ed.extra.doubles[8], framesig_on_at))
                    #print(f"{counter}: {ed.extra.doubles[6]}, {ed.extra.doubles[7]}")
                    pupil_scatter_array[scatter_count,:] = [ed.left.pupil.x, ed.left.pupil.y]
                    cr_scatter_array[scatter_count,:] = [ed.left.cr.x, ed.left.cr.y]
                    p4_scatter_array[scatter_count,:] = [ed.left.p4.x, ed.left.p4.y]
                    scatter_count += 1
                # else:
                #     print('button down no data')
        else:
            if not button_is_pressed:
                button_up = True
    
        counter += 1        


    print(f"Found {counter} frames")
    print(f"Found {scatter_count} button presses")

    return (pupil_array, cr_array, p4_array, button_list)


In [ ]:
def plot_sig(p0, p1, p4, l, n=1):
    #       generate points for plots....
    
    L = len(l)
    datashape = (L,2)
    p0xy = np.zeros(datashape)
    p1xy = np.zeros(datashape)
    p4xy = np.zeros(datashape)
    for i, (ind,fx,fy) in enumerate(l):
        #print(f"{ind}, {fx}, {fy}")
        p0xy[i, :] = np.mean(p0[ind:ind+n, :], axis=0)
        p1xy[i, :] = np.mean(p1[ind:ind+n, :], axis=0)
        p4xy[i, :] = np.mean(p4[ind:ind+n, :], axis=0)


    f, axes = plt.subplots(2, 1, figsize=(10,10))
    axes[0].scatter(p0xy[:L,0], p0xy[:L,1], color='blue', label='pupil')
    axes[0].scatter(p1xy[:L,0], p1xy[:L,1], color='red', label='cr')
    axes[0].scatter(p4xy[:L,0], p4xy[:L,1], color='green', label='p4')
    axes[0].set_xlabel('X')
    axes[0].set_ylabel('Y')
    axes[0].set_title('Scatter Plot of Eye Tracking Data')
    axes[0].legend()

    cr_sig = p1xy-p0xy
    dpi_sig = p4xy-p1xy
    axes[1].scatter(cr_sig[:L,0], cr_sig[:L,1], color='blue', label='CR')
    axes[1].scatter(dpi_sig[:L,0], dpi_sig[:L,1], color='red', label='DPI')
    axes[1].set_xlabel('X')
    axes[1].set_ylabel('Y')
    axes[1].set_title('Raw CR, DPI Tracking Vectors')
    axes[1].legend()

    plt.show()

In [ ]:
plot_sig(pupil_array, cr_array, p4_array, button_list, 100)

In [ ]:
def plot_xy_interval(xy, istart, nbefore, nafter):

    # generate plot x value, call it 't' to eliminate confusion
    t = np.arange(-nbefore, nafter)

    f, ax = plt.subplots(figsize=(10,10))
    
    xcolor = 'tab:red'
    ycolor = 'tab:blue'

    ax.set_xlabel('frame')
    ax.set_ylabel('Xp', color=xcolor)
    ax.tick_params(axis='y', labelcolor=xcolor)
    ax.set_ylim((0,720))

    ax2 = ax.twinx()  # instantiate a second Axes that shares the same x-axis
    ax2.set_ylabel('Yp', color=ycolor)  # we already handled the x-label with ax1
    ax2.tick_params(axis='y', labelcolor=ycolor)
    ax2.set_ylim(0,450)

    plt.axvline(x = 0, color = 'g')

    for i in istart:
        ax.plot(t, xy[i-nbefore:i+nafter,0], color=xcolor)
        ax2.plot(t, xy[i-nbefore:i+nafter,1], color=ycolor)
 
    f.tight_layout()
    plt.show() 


In [ ]:
def plot_from_frame(ax, xy, iframe, ibutton, ntotal,xlim=(0,720),ylim=(0,450), vlinepm=(0,0)):
    """Plot x and y, starting at iframe, with ntotal frames. Vertical line drawn at ibutton, and at vlinepm[0] and [1] below and above, respectively, if those are not (0,0). 

    Args:
        ax (_type_): axis for x
        xy (_type_): axis for y
        iframe (_type_): first frame to plot
        ibutton (_type_): frame where button was hit
        ntotal (_type_): icnlude this many frames
        xlim (tuple, optional): Limits, these are pixel values. Defaults to (0,720).
        ylim (tuple, optional): Limits, pixel values. Defaults to (0,450).
    """

    # generate plot x value, call it 't' to eliminate confusion
    t = np.arange(0, ntotal)
    
    xcolor = 'tab:red'
    ycolor = 'tab:blue'

    ax.set_xlabel('frame')
    ax.set_ylabel('Xp', color=xcolor)
    ax.tick_params(axis='y', labelcolor=xcolor)
    ax.set_ylim(xlim)

    ax2 = ax.twinx()  # instantiate a second Axes that shares the same x-axis
    ax2.set_ylabel('Yp', color=ycolor)  # we already handled the x-label with ax1
    ax2.tick_params(axis='y', labelcolor=ycolor)
    ax2.set_ylim(ylim)

    plt.axvline(x = ibutton-iframe, color = 'g')
    if vlinepm[0] != 0:
        plt.axvline(x = ibutton-iframe-vlinepm[0], color = 'g')
    if vlinepm[1] != 0:
        plt.axvline(x = ibutton-iframe+vlinepm[1], color = 'g')



    ax.plot(t, xy[iframe:iframe+ntotal,0], color=xcolor)
    ax2.plot(t, xy[iframe:iframe+ntotal,1], color=ycolor)
 
    # f.tight_layout()
    # plt.show() 


In [228]:
def plot_velocity(ax, xy, iframe, ibutton, ntotal,xlim=(-50,50),ylim=(-50,50),iframes=(0,0)):

    # generate plot x value, call it 't' to eliminate confusion
    t = np.arange(0, ntotal)
    v = np.diff(xy, axis=0)
    xcolor = 'tab:red'
    ycolor = 'tab:blue'

    ax.set_xlabel('frame')
    ax.set_ylabel('Xp', color=xcolor)
    ax.tick_params(axis='y', labelcolor=xcolor)
    ax.set_ylim(xlim)

    ax2 = ax.twinx()  # instantiate a second Axes that shares the same x-axis
    ax2.set_ylabel('Yp', color=ycolor)  # we already handled the x-label with ax1
    ax2.tick_params(axis='y', labelcolor=ycolor)
    ax2.set_ylim(ylim)

    
    if iframes==(0,0):
        plt.axvline(x = ibutton-iframe, color = 'g')
        ax.plot(t, v[iframe:iframe+ntotal,0], color=xcolor)
        ax2.plot(t, v[iframe:iframe+ntotal,1], color=ycolor)
    else:
        t = np.arange(iframes[0]-ibutton,iframes[1]-ibutton)
        print(f"{iframes[0]},{iframes[1]}")
        ax.plot(t, v[iframes[0]:iframes[1],0], color=xcolor)
        ax2.plot(t, v[iframes[0]:iframes[1],1], color=ycolor)
 
    # f.tight_layout()
    # plt.show() 


In [ ]:
def plot_hist(axX, axY, xy, ibutton, nbefore, nafter, nbins=20):

    xcolor = 'tab:red'
    ycolor = 'tab:blue'

    # Use value at ibutton as center of hist. We take nbins to be both sides, width is 1px. range(xy[ibutton]-nbins, xy[ibusson]+nbins)

    xb = xy[ibutton,0]
    yb = xy[ibutton,1]
    axX.hist(xy[ibutton-nbefore:ibutton+nafter, 0], bins=np.arange(xb-nbins, xb+nbins), color=xcolor)
    axY.hist(xy[ibutton-nbefore:ibutton+nafter, 1], bins=np.arange(yb-nbins, yb+nbins), color=ycolor)


In [220]:


def plot_xyvelh(xy,iframe,ibutton, nframes=1000, nbefore=50,nafter=50,title=""):
    fig = plt.figure(figsize=(12,6))
    if title:
        fig.suptitle(title)
    axs = fig.subplots(1,4,squeeze=True)
    fig.tight_layout()
    plot_from_frame(axs[0], xy, iframe, ibutton, nframes, xlim=(-200,200),ylim=(-200,200), vlinepm=(nbefore,nafter))
    plot_velocity(axs[1], xy, iframe, ibutton, nframes, xlim=(-25,25),ylim=(-25,25), iframes=(ibutton-nbefore,ibutton+nafter))
    plot_hist(axs[2], axs[3], xy, ibutton, nbefore=nbefore, nafter=nafter)
    plt.show()


In [ ]:
def check_velocity(xy, ind, nbefore, nafter, vmax):
    """Check that velocity (in x and y separately) is less than vmax in the range defined by ind, nbefore, and nafter.

    Args:
        xy (numpy array nframes x 2): xy values
        ind (int): index of button press
        nbefore (int): nframes before button press to include in tests
        nafter (int): nframes after button press blah blah
        vmax (float): max velocity 
    """
    v = np.diff(xy, axis=0)
    return np.all(np.abs(v[ind-nbefore:ind+nafter,:]) < vmax)

    

In [249]:
(pupil_array, cr_array, p4_array, button_list) = load_cal_data(filename='./cal91.pkl')
crsig = cr_array - pupil_array
dpisig = p4_array - cr_array


Found 24532 records in ./cal91.pkl
Found 24532 frames
Found 32 button presses


In [250]:

nframes=1000    # for xy and velocity plots, go this many frames from the initial onset frame of the stim
nbefore=50
nafter=50
vmax=10
doplots = False

goodlist=[] # tuples of xav,yav,xv,yv for good trials
# for i in range(8):
#     (ind,vx,vy,f) = button_list[i]
for (ind,vx,vy,f) in button_list:
    # this just checks that there is enough data to plot x and y
    if ind+nframes < np.shape(crsig)[0]:
        # actual data checks to try out
        check1 = check_velocity(crsig,ind,nbefore,nafter,vmax=vmax)
        if check1 and vx<999 and vy < 999:
            m = np.mean(crsig[ind-nbefore:ind+nafter,:], axis=0)    # this is an ndarray
            goodlist.append((m,vx,vy))
            mytitle=f"{vx},{vy},{str(check1)}"
            if doplots:
                plot_xyvelh(crsig, f, ind, nframes=nframes, nbefore=nbefore,nafter=nafter,title=mytitle)
if doplots:
    plt.show()

In [266]:
# Source - https://stackoverflow.com/a/75598551
# Posted by cottontail
# Retrieved 2026-07-08, License - CC BY-SA 4.0

def transform(xy, x_bias, y_bias, x_gain, y_gain, degrees):
    print("transform: ", xy)
    pos = Point(xy[0], xy[1])
    return ((pos + Point(x_bias, y_bias)) * Point(x_gain, y_gain)).rotate(degrees * math.pi / 180)


In [270]:
x_1 = np.linspace(0, 4, 5)
x_2 = np.linspace(2, 10, 5)
print(x_1)
print(np.shape(x_1))
print(x_2)
d=(x_1,x_2)
print(d)
print(type(d))


[0. 1. 2. 3. 4.]
(5,)
[ 2.  4.  6.  8. 10.]
(array([0., 1., 2., 3., 4.]), array([ 2.,  4.,  6.,  8., 10.]))
<class 'tuple'>


In [ ]:
d=np.zeros((len(goodlist),2))
v=np.zeros((len(goodlist),2))
xvalues = []
yvalues = []

for i,t in enumerate(goodlist):
    d[i,:] = t[0]
    v[i,0] = t[1]
    v[i,1] = t[2]
d = d.T # xdata
v = v.T # ydata
p0 = np.array([0,0,1,1,0])

print(np.shape(d))
print(np.shape(v))
print(np.shape(p0))
# 




popt, pcov = curve_fit(transform, d, v, p0=p0)
print(popt)



fig, ax = plt.subplots()
ax.scatter(d[:,0], d[:,1])

ax.set_xlabel(r'X', fontsize=15)
ax.set_ylabel(r'Y', fontsize=15)
ax.set_title('Mean x,y good buttons')

ax.grid(True)
fig.tight_layout()

plt.show()


(2, 30)
(2, 30)
(5,)
transform:  [[ -3.5610614   -4.94213135  -4.68190979  -4.59813812   8.59981964
   -5.13614655 -16.91066315  -3.83682037   9.33780457  -4.5848877
  -16.94595673  -3.63739838   9.57846588  -5.28664795 -16.4707312
   -3.66843933   8.63278809  -4.34841675  -4.43154785  -4.2224884
   -3.15994629  -4.14102997  -4.33514496  -4.58904968   8.88289215
    9.69295685 -15.428461   -16.89311005   8.55033844  10.20167694]
 [ 40.87481979  40.55644821  41.16737289  41.13009521  42.87407349
   40.76338715  39.5664209   40.31761871  41.83333572  41.1822403
   38.99504593  40.57907028  41.83045624  41.15820175  39.66334122
   40.70550674  42.41540421  40.58512405  52.45454422  40.716353
   28.21418304  40.9858638   51.91061005  41.17769424  53.34671143
   30.08268616  27.10417114  50.55414261  53.68989304  29.73738434]]


ValueError: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 30)